# RAG CGU: fluxo completo, local (do banco à avaliação)
Execute as células **em ordem**. Tudo roda na sua máquina: criação do banco SQLite, ingestão dos PDFs, limpeza, enriquecimento,
embeddings, busca, resposta com LLM local e avaliação. A lógica está em `src/cgu_rag/`; aqui você só dispara e inspeciona.

**Antes de começar:** PDFs em `data/pdfs/` e, para a etapa 9, o servidor do LLM local rodando (README, passo 4).
Se mudar a configuração da etapa 1 depois de rodar, **reinicie o kernel** (o módulo legado lê as variáveis ao ser importado).
**Após reiniciar o kernel:** rode só as células 0, 1 (as duas) e **1b (Retomada)**; depois vá direto à etapa 7, 8, 9 ou 10. O banco e os embeddings ficam em disco e não precisam ser refeitos.


## 0. Ambiente
Localiza a raiz do projeto, ajusta o diretório de trabalho e confere as dependências.

In [1]:
import os, sys, importlib.util
from pathlib import Path

# raiz do projeto = pasta que contém pyproject.toml (funciona se o notebook estiver em notebooks/ ou na raiz)
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ / "src"))          # permite rodar sem "pip install -e ."
print("Raiz:", RAIZ, "| Python:", sys.version.split()[0])

faltando = [m for m in ("fitz", "numpy", "requests", "sentence_transformers", "openai") if importlib.util.find_spec(m) is None]
if faltando:
    print("FALTANDO:", faltando)
    print('Instale com:  pip install -e ".[notebook]"   (no terminal, dentro do ambiente virtual) e reinicie o kernel')
else:
    print("Dependências OK")

Raiz: c:\Users\k12_crslcrn\Documents\Projetos\cgu_rag_projeto | Python: 3.11.18
Dependências OK


## 1. Configuração
Caminhos, modelo de embeddings, LLM local e política de sigilo. Edite só esta célula.

In [2]:
import os

os.environ["RAG_DB"] = "data/rag_cgu.db"                          # arquivo do banco (criado na etapa 2)
os.environ["RAG_EMB_MODEL"] = "intfloat/multilingual-e5-small"    # baseline; troque só depois do benchmark
os.environ["LLM_BASE_URL"] = "http://localhost:11434/v1"          # Ollama local
os.environ["LLM_MODEL"] = "jurema-7b"                             # nome criado no "ollama create"
os.environ["SIGILO"] = "sigiloso"                                 # "sigiloso" = só endpoint local (falha fechada)

PASTA_PDFS = "data/pdfs"
GABARITO = "data/gabarito/perguntas.jsonl"

In [3]:
%load_ext autoreload
%autoreload 2
from cgu_rag import config
from cgu_rag.legado import rag_cgu_pipeline as legado
from cgu_rag.busca import buscar, responder
print(config.settings)

Settings(llm_base_url='http://localhost:11434/v1', llm_model='jurema-7b', llm_api_key='ollama', sigilo='sigiloso', db_path='data/rag_cgu.db', emb_model='intfloat/multilingual-e5-small')


## 1b. Retomada (idempotente)
Recria `con`, `llm`, `mostrar` e os imports. Rode depois de reiniciar o kernel; é inofensiva se rodada de novo.

In [4]:
from pathlib import Path
from cgu_rag.llm_local import LLMLocal

Path(legado.DB_PATH).parent.mkdir(parents=True, exist_ok=True)
con = legado.connect()
llm = LLMLocal()          # usa LLM_MODEL=jurema-7b da etapa 1

def mostrar(q, k=5, denso=True):
    for score, r in buscar(con, q, k, denso):
        print(f"[{score:.4f}] {(r['nota_tecnica'] or r['doc_titulo'])[:55]} | {r['titulo'][:70]} (págs {r['pag_ini']}-{r['pag_fim']})")

print("trechos:", con.execute("SELECT COUNT(*) FROM chunks WHERE tipo!='indice'").fetchone()[0],
      "| com vetor:", con.execute("SELECT COUNT(*) FROM chunks WHERE embedding IS NOT NULL").fetchone()[0])

trechos: 1567 | com vetor: 1567


## 2. Criar o banco
`legado.connect()` cria o arquivo SQLite e todo o esquema (`documents`, `chunks`, índice de texto `chunks_fts` com BM25, `meta`).
É idempotente: se o banco já existe, só abre (e migra colunas antigas).

In [5]:
from pathlib import Path
Path(legado.DB_PATH).parent.mkdir(parents=True, exist_ok=True)

con = legado.connect()
print("Banco:", Path(legado.DB_PATH).resolve())
for (nome, tipo) in con.execute("SELECT name, type FROM sqlite_master WHERE type IN ('table','trigger') ORDER BY type, name"):
    print(f"  {tipo:8s} {nome}")

Banco: C:\Users\k12_crslcrn\Documents\Projetos\cgu_rag_projeto\data\rag_cgu.db
  table    chunks
  table    chunks_fts
  table    chunks_fts_config
  table    chunks_fts_data
  table    chunks_fts_docsize
  table    chunks_fts_idx
  table    documents
  table    meta
  trigger  chunks_ad
  trigger  chunks_ai
  trigger  chunks_au


In [6]:
# OPCIONAL: recomeçar do zero (apaga o banco). Descomente para usar.
# con.close(); Path(legado.DB_PATH).unlink(missing_ok=True); con = legado.connect(); print("banco recriado")

## 3. Ingestão dos PDFs
Extrai o texto com PyMuPDF, limpa, segmenta por seções e grava os trechos. PDFs já ingeridos e sem mudança são ignorados (hash SHA-256).

In [7]:
import os, re, time, hashlib
from pathlib import Path
import requests

# ── Configuração ─────────────────────────────────────────────────────────────
os.environ["RAG_DB"] = "data/rag_cgu.db"
os.environ["RAG_EMB_MODEL"] = "intfloat/multilingual-e5-small"
os.environ["LLM_BASE_URL"] = "http://localhost:11434/v1"
os.environ["LLM_MODEL"] = "jurema-7b"
os.environ["SIGILO"] = "sigiloso"

FONTES = [
    "https://basedeconhecimento.cgu.gov.br/bitstreams/ec9c2e26-8443-4cae-9d65-7abea5076ca3/download",
    "https://basedeconhecimento.cgu.gov.br/bitstreams/f1ebf513-6f49-489e-9220-7cfd12b08077/download",
    "https://basedeconhecimento.cgu.gov.br/bitstreams/bcd1fc60-09d0-488a-936b-ab1749ecbfed/download",
    "https://basedeconhecimento.cgu.gov.br/bitstreams/f0476e82-972f-47a0-b7a1-9b4c4d12471f/download",
    "https://basedeconhecimento.cgu.gov.br/bitstreams/0d1ed7f8-6300-4e07-b596-d1d344a1726f/download",
    "https://basedeconhecimento.cgu.gov.br/bitstreams/d699bba4-e046-4e06-ae38-8fdc1e148e60/download",
]
PASTA_PDFS = "data/pdfs"          # se o PDF já estiver aqui, não baixa; senão baixa e salva aqui
GABARITO = "data/gabarito/perguntas.jsonl"

# ── Garante legado e banco (caso as etapas 1 e 2 ainda não tenham sido rodadas) ──
from cgu_rag import config
from cgu_rag.legado import rag_cgu_pipeline as legado
Path(legado.DB_PATH).parent.mkdir(parents=True, exist_ok=True)
try:
    con
except NameError:
    con = legado.connect()
Path(PASTA_PDFS).mkdir(parents=True, exist_ok=True)

# ── Download ou pasta ────────────────────────────────────────────────────────
def arquivo_da_fonte(src: str) -> Path:
    m = re.search(r"/bitstreams/([0-9a-fA-F-]{36})/", src)
    nome = m.group(1) if m else hashlib.sha1(src.encode()).hexdigest()[:12]
    return Path(PASTA_PDFS) / f"{nome}.pdf"

def fetch_pasta_ou_download(src: str) -> bytes:
    if not re.match(r"https?://", src):
        return Path(src).read_bytes()
    arq = arquivo_da_fonte(src)
    if arq.exists() and arq.stat().st_size > 0:
        print(f"  na pasta, sem baixar: {arq.name}")
        return arq.read_bytes()
    ultimo = None
    for tentativa in range(1, 4):
        try:
            r = requests.get(src, timeout=180, headers={"User-Agent": "Mozilla/5.0"}, allow_redirects=True)
            r.raise_for_status()
            arq.write_bytes(r.content)
            print(f"  baixado {len(r.content)/1e6:.1f} MB -> {arq.name}")
            return r.content
        except Exception as e:
            ultimo = e
            time.sleep(2 * tentativa)
    raise RuntimeError(f"falha ao baixar {src}: {ultimo}")

legado.fetch = fetch_pasta_ou_download

nomes_dos_links = {arquivo_da_fonte(u).name for u in FONTES}
locais = [str(p) for p in sorted(Path(PASTA_PDFS).glob("*.pdf")) if p.name not in nomes_dos_links]
fontes = list(dict.fromkeys(FONTES + locais))
print(len(fontes), "fonte(s):", len(FONTES), "link(s) +", len(locais), "PDF(s) local(is) extra(s)")
assert fontes, "Defina links em FONTES ou coloque PDFs em data/pdfs/"

# ── Ingestão ─────────────────────────────────────────────────────────────────
for src in fontes:
    try:
        legado.ingest(con, src)
    except Exception as e:
        print(f"! erro em {src[-60:]}: {e}")
con.commit()

6 fonte(s): 6 link(s) + 0 PDF(s) local(is) extra(s)
  na pasta, sem baixar: ec9c2e26-8443-4cae-9d65-7abea5076ca3.pdf
= ignorado (sem mudanças): https://basedeconhecimento.cgu.gov.br/bitstreams/ec9c2e26-8443-4cae-9d65-7abea5076ca3/download
  na pasta, sem baixar: f1ebf513-6f49-489e-9220-7cfd12b08077.pdf
= ignorado (sem mudanças): https://basedeconhecimento.cgu.gov.br/bitstreams/f1ebf513-6f49-489e-9220-7cfd12b08077/download
  na pasta, sem baixar: bcd1fc60-09d0-488a-936b-ab1749ecbfed.pdf
= ignorado (sem mudanças): https://basedeconhecimento.cgu.gov.br/bitstreams/bcd1fc60-09d0-488a-936b-ab1749ecbfed/download
  na pasta, sem baixar: f0476e82-972f-47a0-b7a1-9b4c4d12471f.pdf
= ignorado (sem mudanças): https://basedeconhecimento.cgu.gov.br/bitstreams/f0476e82-972f-47a0-b7a1-9b4c4d12471f/download
  na pasta, sem baixar: 0d1ed7f8-6300-4e07-b596-d1d344a1726f.pdf
= ignorado (sem mudanças): https://basedeconhecimento.cgu.gov.br/bitstreams/0d1ed7f8-6300-4e07-b596-d1d344a1726f/download
  na pasta, s

In [8]:
for r in con.execute("SELECT id, titulo, n_paginas FROM documents"):
    c = con.execute("SELECT COUNT(*) n, SUM(tipo='secao') s FROM chunks WHERE doc_id=? AND tipo!='indice'", (r["id"],)).fetchone()
    print(f"doc {r['id']}: {r['titulo'][:60]!r} | {r['n_paginas']} págs | {c['n']} trechos | {c['s']} seções")

doc 1: 'EMENTÁRIO DE NOTAS TÉCNICAS Uniformização de entendimentos d' | 211 págs | 247 trechos | 245 seções
doc 2: 'SECRETARIA DE INTEGRIDADE PRIVADA MANUAL DE RESPONSABILIZAÇÃ' | 285 págs | 359 trechos | 342 seções
doc 3: 'MANUAL DE PROCESSO ADMINISTRATIVO DISCIPLINAR (2025)' | 417 págs | 581 trechos | 579 seções
doc 4: 'EMENTÁRIO DE NOTAS TÉCNICAS Uniformização de entendimentos d' | 182 págs | 212 trechos | 211 seções
doc 5: 'EMENTÁRIO DE NOTAS TÉCNICAS Uniformização de entendimentos d' | 111 págs | 164 trechos | 163 seções
doc 6: 'Corregedoria-Geral da União Diretoria de Responsabilização d' | 8 págs | 4 trechos | 0 seções


## 4. Relatório de qualidade da limpeza
Cobertura de páginas, tamanho dos trechos e ruído residual (cabeçalho/rodapé, hífens, caracteres de controle).

In [9]:
for r in legado.quality_report(con):
    print(f"\ndoc {r['doc']}: {r['titulo']!r} | {r['paginas']} págs ({r['paginas_cobertas']} cobertas) | {r['chunks']} trechos "
          f"(mediana {r['tam_mediano']} car., {r['curtos_300']} < 300)")
    print("  ruído residual:", {k: v for k, v in r["ruido"].items()})


doc 1: 'EMENTÁRIO DE NOTAS TÉCNICAS Uniformização de entendimentos d' | 211 págs (211 cobertas) | 247 trechos (mediana 1636 car., 2 < 300)
  ruído residual: {'cabeçalho/rodapé': 0, 'hífen suave': 0, 'caractere de controle': 0, 'pontilhado de sumário': 0, 'palavra partida (hífen no meio do texto)': 0, 'ligadura': 0}

doc 2: 'SECRETARIA DE INTEGRIDADE PRIVADA MANUAL DE RESPONSABILIZAÇÃ' | 285 págs (285 cobertas) | 359 trechos (mediana 2710 car., 6 < 300)
  ruído residual: {'cabeçalho/rodapé': 0, 'hífen suave': 0, 'caractere de controle': 0, 'pontilhado de sumário': 0, 'palavra partida (hífen no meio do texto)': 0, 'ligadura': 0}

doc 3: 'MANUAL DE PROCESSO ADMINISTRATIVO DISCIPLINAR (2025)' | 417 págs (411 cobertas) | 581 trechos (mediana 2572 car., 1 < 300)
  ruído residual: {'cabeçalho/rodapé': 0, 'hífen suave': 0, 'caractere de controle': 0, 'pontilhado de sumário': 0, 'palavra partida (hífen no meio do texto)': 1, 'ligadura': 0}

doc 4: 'EMENTÁRIO DE NOTAS TÉCNICAS Uniformização de 

## 5. Enriquecimento (heurístico, sem LLM)
Gera resumo, palavras-chave, perguntas e contexto por regras. O enriquecimento **por LLM** do módulo legado usa provedores externos,
por isso não é usado aqui enquanto `SIGILO=sigiloso`.

In [10]:
legado.enrich_pending(con, "heuristic")
con.commit()

enriquecimento concluído (heuristic): 0/0 chunks


## 6. Embeddings
Primeira execução baixa o modelo de embeddings (precisa de internet uma vez); depois fica em cache local. O banco trava o modelo usado: para trocar de modelo, limpe os embeddings.

In [11]:
import time
t0 = time.time()
legado.embed_pending(con)
print(f"{time.time()-t0:.0f} s | modelo:", con.execute("SELECT v FROM meta WHERE k='emb_model'").fetchone()["v"])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

embeddings gerados: 0
18 s | modelo: intfloat/multilingual-e5-small


In [12]:
print("trechos:", con.execute("SELECT COUNT(*) FROM chunks WHERE tipo!='indice'").fetchone()[0],
      "| com vetor:", con.execute("SELECT COUNT(*) FROM chunks WHERE embedding IS NOT NULL").fetchone()[0])

trechos: 1567 | com vetor: 1567


## 7. Busca (BM25 + vetores, fusão RRF)

In [13]:
def mostrar(q, k=5, denso=True):
    for score, r in buscar(con, q, k, denso):
        print(f"[{score:.4f}] {(r['nota_tecnica'] or r['doc_titulo'])[:55]} | {r['titulo'][:70]} (págs {r['pag_ini']}-{r['pag_fim']})")

mostrar("prazo para instauração de PAD")

[0.0328] MANUAL DE PROCESSO ADMINISTRATIVO DISCIPLINAR (2025) | Desnecessidade de instauração da sindicância acusatória previamente ao (págs 68-69)
[0.0320] MANUAL DE PROCESSO ADMINISTRATIVO DISCIPLINAR (2025) | Instauração do processo administrativo disciplinar (págs 96-96)
[0.0315] MANUAL DE PROCESSO ADMINISTRATIVO DISCIPLINAR (2025) | Processo administrativo disciplinar (PAD) (págs 73-74)
[0.0307] MANUAL DE PROCESSO ADMINISTRATIVO DISCIPLINAR (2025) | Desnecessidade de instauração da sindicância acusatória previamente ao (págs 69-69)
[0.0303] MANUAL DE PROCESSO ADMINISTRATIVO DISCIPLINAR (2025) | Desnecessidade de instauração da sindicância acusatória previamente ao (págs 68-68)


Comparação rápida: só BM25 contra BM25 + vetores (use perguntas suas):

In [14]:
q = "prazo para instauração de PAD"
print("--- só BM25"); mostrar(q, 5, denso=False)
print("--- BM25 + vetores"); mostrar(q, 5, denso=True)

--- só BM25
[0.0167] MANUAL DE PROCESSO ADMINISTRATIVO DISCIPLINAR (2025) | Desnecessidade de instauração da sindicância acusatória previamente ao (págs 68-69)
[0.0164] MANUAL DE PROCESSO ADMINISTRATIVO DISCIPLINAR (2025) | Desnecessidade de instauração da sindicância acusatória previamente ao (págs 69-69)
[0.0161] MANUAL DE PROCESSO ADMINISTRATIVO DISCIPLINAR (2025) | Instauração do processo administrativo disciplinar (págs 96-96)
[0.0159] MANUAL DE PROCESSO ADMINISTRATIVO DISCIPLINAR (2025) | Processo administrativo disciplinar (PAD) (págs 73-74)
[0.0156] MANUAL DE PROCESSO ADMINISTRATIVO DISCIPLINAR (2025) | Princípio do aproveitamento dos atos processuais (págs 409-410)
--- BM25 + vetores
[0.0328] MANUAL DE PROCESSO ADMINISTRATIVO DISCIPLINAR (2025) | Desnecessidade de instauração da sindicância acusatória previamente ao (págs 68-69)
[0.0320] MANUAL DE PROCESSO ADMINISTRATIVO DISCIPLINAR (2025) | Instauração do processo administrativo disciplinar (págs 96-96)
[0.0315] MANUAL DE PRO

## 8. Testar o LLM local
Se falhar, o servidor não está rodando ou o nome do modelo difere do criado no Ollama.

In [15]:
from cgu_rag.llm_local import LLMLocal
llm = LLMLocal()
try:
    print(llm.chat([{"role": "user", "content": "Responda apenas: pronto."}], max_tokens=10))
    print(llm.medir("Explique em duas frases o que é processo administrativo disciplinar.", max_tokens=64))
except Exception as e:
    print("LLM indisponível:", repr(e))
    print("Confira: 'ollama list' mostra", config.settings.llm_model, "e o servidor está em", config.settings.llm_base_url)

Pronto.
{'ttft_s': 0.13, 'blocos_gerados': 64, 'blocos_por_s': 11.83, 'total_s': 5.54}


## 9. Resposta com citação
Busca os trechos e pede ao LLM uma resposta **só com base neles**, citando documento e páginas.

In [16]:
pergunta = "Quando a prescrição da pretensão punitiva começa a correr?"
out = responder(con, pergunta, k=4, llm=llm)
print(out["resposta"]); print()
for f in out["fontes"]: print("-", f)

Para responder à pergunta sobre quando a prescrição da pretensão punitiva começa a correr, é necessário considerar o contexto legal e jurisprudencial apresentado nos documentos fornecidos.

1. **Princípio da Legalidade e Prescrição**: A prescrição é um instituto do direito que visa garantir a segurança jurídica, evitando que as relações sociais fiquem indefinidamente sem solução. No direito administrativo, a prescrição é a perda do prazo para aplicar penalidades administrativas.

2. **Jurisprudência do STJ**: O Superior Tribunal de Justiça (STJ) estabelece que a contagem do prazo prescricional não se inicia com a mera ocorrência do ilícito, mas sim na data em que a autoridade competente toma conhecimento do fato. Isso é fundamentado na Súmula 635 do STJ, que afirma que a prescrição não corre contra quem não pode agir.

3. **Cenários de Sigilo e Investigação**: Em casos onde há investigações criminais protegidas por sigilo judicial ou indisponibilidade de provas, a Administração Pública

## 10. Avaliação da recuperação (hit@k e MRR)
Crie `data/gabarito/perguntas.jsonl`, uma linha por pergunta: `{"id": "q01", "pergunta": "...", "esperado": "Nota Técnica nº ..."}`.
A célula cria um modelo com exemplos **fictícios** se o arquivo não existir; substitua pelo seu benchmark.

In [17]:
import json
g = Path(GABARITO); g.parent.mkdir(parents=True, exist_ok=True)
if not g.exists():
    exemplo = [{"id": "q01", "pergunta": "Qual o prazo para instauração de PAD?", "esperado": "TROQUE-PELA-NOTA-TECNICA-ESPERADA"}]
    g.write_text("\n".join(json.dumps(x, ensure_ascii=False) for x in exemplo), encoding="utf-8")
    print("modelo criado em", g, "-> edite antes de avaliar")

modelo criado em data\gabarito\perguntas.jsonl -> edite antes de avaliar


In [18]:
from cgu_rag.avaliacao import avaliar_recuperacao
for denso in (False, True):
    for k in (1, 3, 5):
        print("BM25+vetores" if denso else "só BM25", k, avaliar_recuperacao(GABARITO, con, k=k, denso=denso))

só BM25 1 {'n': 1, 'hit@1': 0.0, 'mrr@1': 0.0}
só BM25 3 {'n': 1, 'hit@3': 0.0, 'mrr@3': 0.0}
só BM25 5 {'n': 1, 'hit@5': 0.0, 'mrr@5': 0.0}
BM25+vetores 1 {'n': 1, 'hit@1': 0.0, 'mrr@1': 0.0}
BM25+vetores 3 {'n': 1, 'hit@3': 0.0, 'mrr@3': 0.0}
BM25+vetores 5 {'n': 1, 'hit@5': 0.0, 'mrr@5': 0.0}


Registre os números do baseline (A). Qualquer mudança futura (novo embedding, reranker, regex de metadados) deve ser comparada com eles.